In [1]:
import numpy as np


def designer_percentile(scores, grid_size, a=7.0, b=3.0, m0=0.40, s0=0.30, sigma_ref=4.0):
    H, W = grid_size
    Smax = H * W
    scores = np.asarray(scores, dtype=float)
    mu, sigma = scores.mean(), scores.std()
    m = mu / Smax
    s = min(sigma, Smax/2) / (Smax/2)
    # Step A: 难度 & 区分度
    z = a * (m0 - m) + b * (s - s0)
    p = 1.0 / (1.0 + np.exp(-z))
    # Step B: 竞争性压缩
    kappa = min(1.0, sigma / sigma_ref)
    p_prime = 0.5 + kappa * (p - 0.5)
    return p_prime

def designer_in_game_score(scores, p_prime, bump_top=False, eps=1e-6):
    scores = np.sort(np.asarray(scores, dtype=float))
    n = len(scores)
    pis = (np.arange(1, n+1) - 0.5) / n
    val = float(np.interp(p_prime, pis, scores))
    if bump_top and p_prime >= pis[-1]:
        val = scores[-1] + eps
    return val

def designer_meta_score(p_prime, grid_size):
    H, W = grid_size
    return (H * W) * p_prime  # ∈ [0, S_max]


In [16]:
grid = (6, 6)

games = {
    "Case A (Easy & non-discriminative)": [15,14,16,15],
    "Case B (Hard & mildly discriminative": [5,8,10,3],
    "Case C (Hard & highly discriminative)": [-5, -3, 15,20],
}

for name, scores in games.items():
    p_prime = designer_percentile(scores, grid)
    in_game = designer_in_game_score(scores, p_prime, bump_top=True)
    meta = designer_meta_score(p_prime, grid)

    combined = np.append(scores, in_game)
    rank = (len(combined) - np.argsort(np.argsort(combined))[-1])  # rank 1 = highest

    print(f"{name}")
    print(f"Players: {scores}")
    print(f"→ Percentile (p'): {p_prime:.3f}")
    print(f"→ In-game Designer Score: {in_game:.2f}")
    print(f"→ Rank among players: #{rank}")
    print(f"→ Meta Designer Score: {meta:.2f}\n")

Case A (Easy & non-discriminative)
Players: [15, 14, 16, 15]
→ Percentile (p'): 0.463
→ In-game Designer Score: 15.00
→ Rank among players: #2
→ Meta Designer Score: 16.66

Case B (Hard & mildly discriminative
Players: [5, 8, 10, 3]
→ Percentile (p'): 0.667
→ In-game Designer Score: 8.33
→ Rank among players: #2
→ Meta Designer Score: 24.00

Case C (Hard & highly discriminative)
Players: [-5, -3, 15, 20]
→ Percentile (p'): 0.917
→ In-game Designer Score: 20.00
→ Rank among players: #1
→ Meta Designer Score: 33.03

